# HyperPod IAM — Job Role Auto-Creation & Caller-Side CLI Permission Check (No Mocks)

HyperPod involves **two distinct IAM identities**, and this notebook demonstrates both against **real AWS**:

| Identity | Who/what it is | What it needs | How the SDK handles it |
|---|---|---|---|
| **Job execution role** | Assumed by the training job **on the cluster** (trusted by `sagemaker.amazonaws.com`) | S3 / ECR / CloudWatch / KMS (job runtime) | Auto-created as `SageMaker-AutoRole-HyperPod` via the resolver |
| **Caller identity** | Whoever runs the **HyperPod CLI** locally (`connect-cluster` + `start-job`) | `sagemaker:DescribeCluster`, `eks:DescribeCluster`, `eks:AccessKubernetesApi` | **Verified** on the caller (cannot be auto-created); warns if missing |

The connect permissions are deliberately **not** attached to the SageMaker-trusted job role — the local caller can't assume that role, so granting them there would be useless.

> ⚠️ **Real AWS.** Provisioning the job role creates an IAM role + policies in the active account. The final cell deletes everything it created.

> 🐍 **Kernel:** requires **Python 3.10+** (PEP 604 typing). Use the `Python (sagemaker-recipes)` kernel.

## Setup

In [ ]:
import sys

assert sys.version_info >= (3, 10), (
    f"Needs Python 3.10+ (got {sys.version.split()[0]}). "
    "Switch to the 'Python (sagemaker-recipes)' kernel."
)

for p in (
    "../sagemaker-core/src",
    "../sagemaker-train/src",
    "../sagemaker-serve/src",
    "../sagemaker-mlops/src",
):
    if p not in sys.path:
        sys.path.insert(0, p)

import logging
import boto3
from botocore.exceptions import ClientError

logging.getLogger().setLevel(logging.WARNING)  # quiet SDK info logs

iam = boto3.client("iam")
sts = boto3.client("sts")
REGION = boto3.Session().region_name or "us-west-2"
ACCOUNT = sts.get_caller_identity()["Account"]
HP_ROLE = "SageMaker-AutoRole-HyperPod"

print(f"Account:  {ACCOUNT}")
print(f"Identity: {sts.get_caller_identity()['Arn']}")
print(f"Region:   {REGION}")
print("\n→ Confirm this is a NON-PRODUCTION account before continuing.")

In [ ]:
def role_exists(name):
    try:
        iam.get_role(RoleName=name)
        return True
    except ClientError:
        return False


def show_role(name):
    assert role_exists(name), f"❌ Role {name} was NOT created!"
    role = iam.get_role(RoleName=name)["Role"]
    principal = role["AssumeRolePolicyDocument"]["Statement"][0]["Principal"]
    policies = [p["PolicyName"] for p in iam.list_attached_role_policies(RoleName=name)["AttachedPolicies"]]
    print(f"✓ Role: {name}")
    print(f"  Trust principal: {principal}")
    print(f"  Policies ({len(policies)}):")
    for pn in sorted(policies):
        print(f"    - {pn}")


def cleanup_role(name):
    try:
        for p in iam.list_attached_role_policies(RoleName=name).get("AttachedPolicies", []):
            iam.detach_role_policy(RoleName=name, PolicyArn=p["PolicyArn"])
            if f":{ACCOUNT}:policy/" in p["PolicyArn"]:
                try:
                    for v in iam.list_policy_versions(PolicyArn=p["PolicyArn"]).get("Versions", []):
                        if not v["IsDefaultVersion"]:
                            iam.delete_policy_version(PolicyArn=p["PolicyArn"], VersionId=v["VersionId"])
                    iam.delete_policy(PolicyArn=p["PolicyArn"])
                except ClientError:
                    pass
        iam.delete_role(RoleName=name)
        print(f"  cleaned: {name}")
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchEntity":
            print(f"  warn {name}: {e}")

## 1. Provision the HyperPod **job execution role**

`resolve_or_create_role(role_type="hyperpod")` is the core entry point that provisions the HyperPod **job execution role**. With no role provided and a caller lacking the job-runtime permissions, it auto-creates `SageMaker-AutoRole-HyperPod` (trusted by `sagemaker.amazonaws.com`) with least-privilege S3/ECR/CloudWatch/KMS policies. This is the role a HyperPod job assumes on the cluster; it is provisioned separately from the caller-side CLI permission check shown in section 2.

> If your caller identity already has those permissions, the resolver will reuse it instead of creating a new role — that's expected.

In [ ]:
cleanup_role(HP_ROLE)  # start fresh so the role below is genuinely created here
assert not role_exists(HP_ROLE)

from sagemaker.core.helper.iam_role_resolver import resolve_or_create_role

role_arn = resolve_or_create_role(provided_role=None, role_type="hyperpod")
print(f"\nResolved role: {role_arn}\n")

if HP_ROLE in role_arn:
    show_role(HP_ROLE)
else:
    print(f"(Caller identity already had HyperPod job-runtime perms; reused {role_arn} "
          f"instead of creating {HP_ROLE}.)")

### The job role excludes the CLI connect permissions

Confirm the auto-created job role carries **only** job-runtime permissions and does **not** include the EKS/cluster-connect actions — those belong on the caller.

In [ ]:
from sagemaker.core.helper.iam_role_resolver import (
    _get_required_actions,
    HYPERPOD_CLI_CONNECT_ACTIONS,
)

job_actions = set(_get_required_actions("hyperpod"))
print("Job-role actions include S3/ECR/logs:",
      {"s3:GetObject", "ecr:BatchGetImage", "logs:CreateLogStream"}.issubset(job_actions))
print("Job-role EXCLUDES CLI connect actions:",
      all(a not in job_actions for a in HYPERPOD_CLI_CONNECT_ACTIONS))
print("\nCLI connect actions (verified on the caller, not on this role):")
for a in HYPERPOD_CLI_CONNECT_ACTIONS:
    print("   -", a)

## 2. Verify the **caller-side** HyperPod CLI permissions

`verify_hyperpod_connect_permissions()` simulates the connect actions against the **caller** identity (via `iam:SimulatePrincipalPolicy`) and warns — non-blocking — if any are missing. This is exactly what `BaseTrainer._train_hyperpod` runs before invoking the HyperPod CLI.

Returns: `True` (all allowed) / `False` (one denied → warning logged) / `None` (couldn't determine, e.g. caller is an IAM user or can't self-simulate).

In [ ]:
import logging as _logging
_logging.getLogger("sagemaker.core.helper.iam_role_resolver").setLevel(_logging.WARNING)

from sagemaker.core.helper.iam_role_resolver import verify_hyperpod_connect_permissions

verdict = verify_hyperpod_connect_permissions(cluster_name="my-hyperpod-cluster")
print(f"\nConnect-permission verdict: {verdict}")
print({True: "Caller can run the HyperPod CLI.",
       False: "Caller is MISSING connect permissions (see warning above).",
       None: "Undetermined (IAM user / cannot self-simulate)."}[verdict])

## 3. (Illustrative) Trainer call with `HyperPodCompute`

This is how a user would target HyperPod. The trainer dispatches to `_train_hyperpod`, which runs the caller-side check above and then shells out to the `hyperpod` CLI. Without the CLI installed / a real cluster this won't submit — the point is the IAM wiring, already shown in cells 1–2. Left here as a reference snippet (not executed).

In [ ]:
# from sagemaker.train.sft_trainer import SFTTrainer
# from sagemaker.core.training.configs import HyperPodCompute
#
# trainer = SFTTrainer(
#     model="amazon.nova-lite-v2",
#     compute=HyperPodCompute(cluster_name="my-hyperpod-cluster", instance_type="ml.p5.48xlarge"),
#     model_package_group="my-group",
#     s3_output_path=f"s3://sagemaker-{REGION}-{ACCOUNT}/hp-demo/",
# )
# trainer.train(training_dataset="s3://my-bucket/train.jsonl", wait=False)
print("Reference snippet only — see cells 1 and 2 for the executed IAM behavior.")

## Cleanup
Remove the auto-created job role + its policies and verify they are gone.

In [ ]:
print("Removing auto-created HyperPod job role...")
cleanup_role(HP_ROLE)
print("\nVerifying...")
print(f"  {'✓ deleted' if not role_exists(HP_ROLE) else '✗ still exists!'}: {HP_ROLE}")